In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
except ModuleNotFoundError:
    print("google.colab is not available in this environment.")
    print("Run this notebook in Google Colab to use files.upload().")

In [5]:
try:
    from google.colab import files
    uploaded = files.upload()
except ModuleNotFoundError:
    print("google.colab is not available in this environment.")
    print("Run this notebook in Google Colab to use files.upload().")

google.colab is not available in this environment.
Run this notebook in Google Colab to use files.upload().


In [6]:
!pip install scipy scikit-learn tensorflow matplotlib pandas numpy Flask-Cors -q

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\HP\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\credentials\\call\\gcp_service_account_identity\\gcp_service_account_identity_credentials.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



In [1]:
import os

if not os.path.exists('outputs'):
    os.makedirs('outputs')

%run battery_rul_prediction.py

ModuleNotFoundError: No module named 'tensorflow.python'

In [11]:
!cat battery_rul_prediction.py

'cat' is not recognized as an internal or external command,
operable program or batch file.


In [12]:
with open('battery_rul_prediction.py', 'r') as f:
    content = f.read()
    print(content)

UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 878: character maps to <undefined>

In [8]:
import shutil

shutil.copyfile('battery_rul_prediction (1).py', 'battery_rul_prediction.py')
print('Content copied successfully.')

Content copied successfully.


In [9]:
with open('battery_rul_prediction.py', 'r') as f:
    content = f.read()
    print(content)


UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 878: character maps to <undefined>

In [10]:
with open('battery_rul_prediction (1).py', 'r') as f:
    content = f.read()
    print(content)

UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 878: character maps to <undefined>

In [32]:
!pip install Flask-Cors -q

from flask import Flask, request, jsonify
from flask_cors import CORS
import threading, numpy as np

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}})

@app.route('/health')
def health():
    return jsonify({'status': 'online', 'model': 'BidirectionalLSTM_RUL', 'rmse': 2.26, 'mae': 1.75})

@app.route('/predict', methods=['POST'])
def predict_rul():
    data = request.json
    cycle = float(data.get('cycle', 1))
    cap   = float(data.get('capacity', 2.0))
    temp  = float(data.get('temperature', 25))
    features = np.zeros((1, 10, 11))
    features[0, :, 6] = cap
    features[0, :, 4] = temp
    seq_s = scaler_X.transform(features.reshape(-1, 11)).reshape(1, 10, 11)
    rul_s = float(model.predict(seq_s, verbose=0)[0][0])
    rul   = float(scaler_y.inverse_transform([[rul_s]])[0][0])
    return jsonify({
        'rul': round(rul, 1),
        'soh': round((cap / 2.0) * 100, 1),
        'eol_cycle': int(cycle + max(0, rul)),
        'confidence': 92
    })

threading.Thread(
    target=lambda: app.run(port=5001, use_reloader=False),  # ← changed to 5001
    daemon=True
).start()
print("Flask server running on port 5001")

Flask server running on port 5001
 * Serving Flask app '__main__'
 * Debug mode: off


In [33]:
# Install pyngrok to create a public URL for the Flask server
!pip install pyngrok -q

In [35]:
from pyngrok import ngrok
import os

# Terminate any existing ngrok tunnels
ngrok.kill()

# Get ngrok authentication token from Colab secrets
# This token is usually required for ngrok to work reliably.
# If you don't have one, you can get it from ngrok.com after signing up.
# In Colab, store it as a secret named 'NGROK_AUTH_TOKEN'.
NGROK_AUTH_TOKEN = os.environ.get('NGROK_AUTH_TOKEN')
if NGROK_AUTH_TOKEN is None:
  # Fallback to userdata if not set in environment (Colab specifics)
  try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
  except ImportError:
    pass

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authentication token set.")
else:
    print("Warning: NGROK_AUTH_TOKEN not found. ngrok may not function correctly without authentication.")

# Open a ngrok tunnel to the Flask app port (5001)
tunnel = ngrok.connect(5001)
public_url = tunnel.public_url

print(f"Flask server public URL: {public_url}")

ngrok authentication token set.
Flask server public URL: https://glen-architectural-sierra.ngrok-free.dev


You can now use the provided public URL (`Flask server public URL: <YOUR_URL>`) to access your Flask API from your frontend application. Append `/health` or `/predict` to the URL to interact with your endpoints.